# UR3e base-frame calibration (mocap → robot base)

Recover the UR3e base frame **{B}** — origin `p0` and orientation `R0` — in the
motion-capture world frame **{W}**, using only what the cameras can see.

**Method (all closed form, no iterative regression):**
1. Sweep **joint 1** alone → the tracked rigid body traces a circle; its axis is
   the joint-1 (vertical) rotation axis.
2. Sweep **joint 2** alone → a second circle; its axis is the joint-2 axis.
3. The two joint axes intersect at the **shoulder** point → intersect the lines.
4. `p0 = shoulder − d1 · z0`  (`d1 = 0.15185 m`, `z0` = joint-1 axis up).
5. Two orientation-fixed base-frame probe moves (**+X**, **+Y**) give the base
   axes in {W} → orthonormalize → `R0`.

A base-frame point maps to world by `p_W = p0 + R0 @ p_B` (R0's columns are the
base axes in {W}). Output → `robots/UR3e/base_frame_calibration.json`.

> **Run Phase 1 first.** It is pure numpy (no hardware) and must recover a known
> frame to **< 1 mm** before you touch the robot.

## Setup

In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

# Locate this calibration dir (kernel cwd may be the repo root or here).
_CANDIDATES = [os.getcwd(),
               os.path.join(os.getcwd(), "robots", "UR3e", "calibration")]
_HERE = next((c for c in _CANDIDATES
              if os.path.exists(os.path.join(c, "base_calibration_dependencies.py"))), None)
if _HERE is None:
    raise FileNotFoundError("base_calibration_dependencies.py not found near cwd")
sys.path.insert(0, _HERE)

import base_calibration_dependencies as cal

print("solver loaded from", cal.__file__)
print("d1 =", cal.UR3E_D1, "m")

## Phase 1 — synthetic check (no hardware)

Build two circles that share the shoulder point (so the joint axes truly
intersect), add 0.5 mm/point noise, and confirm `solve()` recovers the known
frame. These arrays also serve as the **dry-run** data for Phase 2.

In [ ]:
def _rot_about(axis, ang):
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    x, y, z = axis
    c, s, C = np.cos(ang), np.sin(ang), 1 - np.cos(ang)
    return np.array([
        [c + x*x*C, x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s, c + y*y*C, y*z*C - x*s],
        [z*x*C - y*s, z*y*C + x*s, c + z*z*C]])

def _plane_basis(n):
    n = n / np.linalg.norm(n)
    a = np.array([1.,0.,0.]) if abs(n[0]) < 0.9 else np.array([0.,1.,0.])
    v1 = a - (a @ n) * n; v1 /= np.linalg.norm(v1)
    return v1, np.cross(n, v1)

rng = np.random.default_rng(0)
NOISE = 0.0005  # 0.5 mm/point
d1 = cal.UR3E_D1

# Ground-truth base frame in the world frame.
p0_true = np.array([0.612, -0.345, 0.018])
R0_true = _rot_about([0,0,1], np.deg2rad(37.0)) @ _rot_about([1,0,0], np.deg2rad(2.0))
z0_true, y0_true = R0_true[:, 2], R0_true[:, 1]
shoulder = p0_true + d1 * z0_true

# Joint-1 circle (axis z0 through the base/shoulder line).
c1 = shoulder + 0.22 * z0_true; r1 = 0.18
va, vb = _plane_basis(z0_true)
th = np.deg2rad(np.linspace(-85, 85, 40))
pts1 = c1 + r1*(np.cos(th)[:,None]*va + np.sin(th)[:,None]*vb)
pts1 = pts1 + rng.normal(0, NOISE, pts1.shape)

# Joint-2 circle (axis y0 through the SAME shoulder point).
c2 = shoulder + 0.10 * y0_true; r2 = 0.25
va, vb = _plane_basis(y0_true)
th = np.deg2rad(np.linspace(-80, 80, 40))
pts2 = c2 + r2*(np.cos(th)[:,None]*va + np.sin(th)[:,None]*vb)
pts2 = pts2 + rng.normal(0, NOISE, pts2.shape)

# Probe moves: +X, +Y in base frame, length L (orientation fixed).
L = 0.10
dp_x = R0_true @ np.array([L,0,0]) + rng.normal(0, NOISE, 3)
dp_y = R0_true @ np.array([0,L,0]) + rng.normal(0, NOISE, 3)
print(f"synthetic: {len(pts1)} J1 pts, {len(pts2)} J2 pts, L={L} m, noise={NOISE*1e3} mm")

In [ ]:
# Solve and check recovery against ground truth.
p0, R0, res = cal.solve(pts1, pts2, dp_x, dp_y, L, d1)

p0_err_mm = np.linalg.norm(p0 - p0_true) * 1e3
ang = np.degrees(np.arccos(np.clip((np.trace(R0.T @ R0_true) - 1)/2, -1, 1)))
print(f"p0_true  = {np.round(p0_true, 4)}")
print(f"p0_recov = {np.round(p0, 4)}")
print(f"p0 error          = {p0_err_mm:.3f} mm")
print(f"R0 geodesic error = {ang:.3f} deg")
print("residuals:")
for k, v in res.items():
    print(f"  {k:22s} = {v:8.4f}")

assert p0_err_mm < 1.0, f"p0 error {p0_err_mm:.3f} mm exceeds 1 mm"
assert ang < 1.0, f"R0 error {ang:.3f} deg too large"
print("\nPHASE 1 PASSED: p0 < 1 mm.")

In [ ]:
# Cross-check: full SVD+Kasa fit vs the 3-point circumcenter (circle 1).
c_fit, n_fit, rad_fit, _ = cal.fit_circle_3d(pts1)
idx = [0, len(pts1)//2, len(pts1)-1]
c_3pt, n_3pt = cal.circumcenter_3pt(*[pts1[i] for i in idx])
print(f"full-fit center  = {np.round(c_fit, 4)}  (r={rad_fit:.4f})")
print(f"3-point center   = {np.round(c_3pt, 4)}")
print(f"center disagreement = {np.linalg.norm(c_fit - c_3pt)*1e3:.3f} mm")
print(f"axis angle diff     = {np.degrees(np.arccos(np.clip(abs(n_fit@n_3pt),-1,1))):.3f} deg")

In [ ]:
# Visualize: the two circles, their axes, the shoulder, and p0.
c1f, a1f, r1f, _ = cal.fit_circle_3d(pts1)
c2f, a2f, r2f, _ = cal.fit_circle_3d(pts2)
P1, P2, gap = cal.closest_point_two_lines(c1f, a1f, c2f, a2f)

fig = plt.figure(figsize=(8, 7)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(*pts1.T, s=8, label="joint-1 sweep")
ax.scatter(*pts2.T, s=8, label="joint-2 sweep")
for c, a, col in [(c1f, a1f, "C0"), (c2f, a2f, "C1")]:
    seg = np.stack([c - 0.3*a, c + 0.3*a])
    ax.plot(*seg.T, col, lw=2)
ax.scatter(*P1, c="k", s=60, marker="x", label=f"shoulder (gap {gap*1e3:.2f} mm)")
ax.scatter(*p0, c="r", s=90, marker="*", label="p0 (base origin)")
ax.scatter(*p0_true, c="g", s=40, marker="o", label="p0 true")
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z"); ax.legend(fontsize=8)
ax.set_title("Phase 1 synthetic: circles, axes, shoulder, p0"); plt.tight_layout(); plt.show()

---
## Phase 2 — hardware

**Safety / procedure**
- `DRY_RUN = True` reuses the Phase 1 synthetic arrays so the whole pipeline runs
  with **no robot and no mocap**. Set it `False` only at the robot, after Phase 1
  passes.
- The robot is gated: `connect()` **blocks until the External Control program is
  PLAYING** on the pendant (pressing **Play** is the robot-side "go").
- **Only one joint moves per sweep.** Low speed/accel. Keep `calibrig4` off both
  rotation axes and camera-visible across the whole sweep.
- The orientation probe uses the **raw** RTDE TCP pose (`getActualTCPPose()`),
  NOT `receive_feedback()` — the latter negates X/Y for the policy obs frame.

In [ ]:
# --- CONFIG ---
DRY_RUN = True                 # <-- set False at the robot (after Phase 1 passes)

ROBOT_IP        = "192.168.1.4"     # UR3e on PolyScope X
USE_EXT_URCAP   = True               # External Control URCapX go-gate
UR_CAP_PORT     = 50002

MOCAP_SERVER_IP = "10.1.1.198"
RIGID_BODY      = "calibrig4"        # tracked rigid body on the arm

# Safe start config (rad), seeded from the UR3 'task_home' keyframe
# (mjx_single_cube_position_ur3.xml): arm reaches UP-and-OUT, so calibrig4 sits
# at a large radius from BOTH the vertical joint-1 axis and the horizontal
# shoulder axis (big, well-conditioned circles) and stays camera-visible.
# Verify visibility at the robot and jog if a camera is occluded, then re-read q.
Q_SAFE_START = np.array([0.0, -2.0, 1.6, -1.6, -1.5, 0.0])   # task_home arm joints

SWEEP_DEG   = 160.0            # half-circle span per joint (150-180)
SWEEP_V     = 0.12            # rad/s  (SLOW)
SWEEP_A     = 0.4            # rad/s^2
PROBE_L     = 0.10            # m, probe move length (orientation fixed)
PROBE_V     = 0.03            # m/s
PROBE_A     = 0.2            # m/s^2

OUT_JSON = os.path.join(os.path.dirname(_HERE), "base_frame_calibration.json")
print("DRY_RUN =", DRY_RUN, "| output ->", OUT_JSON)

In [ ]:
# Connect mocap (skipped in DRY_RUN). Verify data is flowing and units are meters.
reader = None
if not DRY_RUN:
    sys.path.insert(0, os.path.join(os.getcwd(), "motion_capture", "mymocap"))
    from vrpn_dependencies import VRPNRigidBodyReader
    reader = VRPNRigidBodyReader(MOCAP_SERVER_IP, rigid_body_name=RIGID_BODY,
                                 names=[RIGID_BODY])
    assert reader.start(timeout=8.0), "VRPN subscribe failed"
    assert reader.wait_for_data(timeout=5.0), f"no reports for '{RIGID_BODY}'"
    xyz = reader.get_rigid_body_xyz()
    print(f"'{RIGID_BODY}' @ {np.round(xyz, 3)} m  (|xyz|={np.linalg.norm(xyz):.3f} m)")
    assert np.linalg.norm(xyz) < 10.0, "xyz looks like mm, not meters!"
else:
    print("DRY_RUN: skipping mocap connect.")

In [ ]:
# Connect robot + go-gate (skipped in DRY_RUN). Blocks until Play on the pendant.
robot = None
if not DRY_RUN:
    sys.path.insert(0, os.path.dirname(_HERE))  # robots/UR3e
    from ur3_realrobot_dependencies import UR3RealRobotPick
    robot = UR3RealRobotPick(host=ROBOT_IP, use_ext_urcap=USE_EXT_URCAP,
                             ur_cap_port=UR_CAP_PORT)
    print("Press PLAY on the pendant (External Control) to release the go-gate...")
    robot.connect()
    robot.print_feedback()
    print("connected:", robot.is_connected())
else:
    print("DRY_RUN: skipping robot connect.")

In [ ]:
# Move to the safe start (skipped in DRY_RUN).
if not DRY_RUN:
    robot.move_to_start(Q_SAFE_START, a=0.5, v=0.1)
    assert reader.get_rigid_body_xyz() is not None, "calibrig4 not visible at start!"
    print("at safe start; calibrig4 visible.")
else:
    print("DRY_RUN: skipping move_to_start.")

In [ ]:
# Recording helpers. sweep_joint moves ONE joint async and logs (mocap pt, q)
# while it travels; probe_axis does an orientation-fixed Cartesian move.
def sweep_joint(joint_idx, q_start, delta, v, a, hz=30.0, timeout=60.0):
    q_start = np.asarray(q_start, float)
    q_goal = q_start.copy(); q_goal[joint_idx] += delta
    robot.send_movej(q_goal, a=a, v=v)            # async
    pts, qs = [], []
    t0 = time.perf_counter()
    while True:
        fb = robot.receive_feedback()
        xyz = reader.get_rigid_body_xyz()
        if xyz is not None:
            pts.append(xyz); qs.append(fb["q"])
        if np.linalg.norm(np.array(fb["q"]) - q_goal) < 0.01:
            break
        if time.perf_counter() - t0 > timeout:
            print("  sweep timeout"); break
        time.sleep(1.0 / hz)
    return np.array(pts), np.array(qs)

def probe_axis(axis_idx, L, v, a, settle=0.6):
    r = robot.connect()
    p_before = reader.get_rigid_body_xyz().copy()
    pose0 = r.getActualTCPPose()                  # RAW base-frame pose
    pose1 = list(pose0); pose1[axis_idx] += L
    robot._control.moveL(pose1, v, a)             # blocking, orientation fixed
    time.sleep(settle)
    p_after = reader.get_rigid_body_xyz().copy()
    robot._control.moveL(list(pose0), v, a)       # return to start
    return p_after - p_before

print("helpers defined.")

In [ ]:
# Joint-1 half-circle (hold q2..q6). DRY_RUN keeps the Phase 1 array.
if not DRY_RUN:
    pts1, q1log = sweep_joint(0, Q_SAFE_START, np.deg2rad(SWEEP_DEG), SWEEP_V, SWEEP_A)
    print(f"joint-1: recorded {len(pts1)} points")
else:
    print(f"DRY_RUN: using synthetic pts1 ({len(pts1)} points)")

In [ ]:
# Joint-2 half-circle (do it at q1=0; hold the rest). DRY_RUN keeps Phase 1 array.
if not DRY_RUN:
    q2_base = Q_SAFE_START.copy(); q2_base[0] = 0.0
    robot.move_to_start(q2_base, a=0.5, v=0.1)
    pts2, q2log = sweep_joint(1, q2_base, np.deg2rad(SWEEP_DEG), SWEEP_V, SWEEP_A)
    print(f"joint-2: recorded {len(pts2)} points")
else:
    print(f"DRY_RUN: using synthetic pts2 ({len(pts2)} points)")

In [ ]:
# Orientation probe (Option A): +X then +Y, orientation fixed. DRY_RUN keeps arrays.
if not DRY_RUN:
    robot.move_to_start(Q_SAFE_START, a=0.5, v=0.1)
    dp_x = probe_axis(0, PROBE_L, PROBE_V, PROBE_A)
    dp_y = probe_axis(1, PROBE_L, PROBE_V, PROBE_A)
    print(f"dp_x = {np.round(dp_x,4)} (|{np.linalg.norm(dp_x):.4f}|)")
    print(f"dp_y = {np.round(dp_y,4)} (|{np.linalg.norm(dp_y):.4f}|)")
else:
    print("DRY_RUN: using synthetic dp_x, dp_y")

In [ ]:
# Solve from the (live or dry-run) data and write the JSON.
import json
from datetime import datetime

p0, R0, res = cal.solve(pts1, pts2, dp_x, dp_y, PROBE_L if not DRY_RUN else L, d1)
out = cal.build_output_dict(
    p0, R0, res, n_points_j1=len(pts1), n_points_j2=len(pts2),
    rigid_body=RIGID_BODY, d1=d1, timestamp=datetime.now().isoformat(timespec="seconds"))

print("p0 (m) =", np.round(p0, 4))
print("R0 =\n", np.round(R0, 4))
print("residuals:")
for k, v in res.items():
    print(f"  {k:22s} = {v:8.4f}")
print(f"\naxis gap h = {res['axis_gap_h_mm']:.2f} mm   (smaller = better)")

if not DRY_RUN:
    with open(OUT_JSON, "w") as f:
        json.dump(out, f, indent=2)
    print("wrote", OUT_JSON)
else:
    print("DRY_RUN: not writing JSON. Preview:")
    print(json.dumps(out, indent=2))

In [ ]:
# Sanity check: p0 vs where the robot stands. The base origin should be near the
# floor (small Z in {W} if the world Z is up) and within the workspace footprint.
print(f"p0 = {np.round(p0, 4)} m")
print(f"shoulder height recovered along z0: d1 used = {d1:.5f} m")
if not DRY_RUN:
    print("Compare p0 to a tape-measure estimate of the base origin in mocap coords.")
    robot.disconnect(); reader.stop()
    print("disconnected.")